In [2]:
import sys, duckdb, pandas as pd
from pathlib import Path
ROOT = Path("/Users/shraddha/Documents/DATA_SCIENCE/Shellscope/shellscope")
sys.path.insert(0, str(ROOT / "src"))
from shellscope.matching import normalise_name

OS_CSV = ROOT / "data" / "raw" / "opensanctions" / "targets.simple.csv"
con = duckdb.connect(str(ROOT / "data" / "shellscope.duckdb"))
def q(sql): return con.sql(sql).df()
SRC = f"read_csv('{OS_CSV}', header = true, all_varchar = true)"

print(q(f"DESCRIBE SELECT * FROM {SRC}")[["column_name"]].T)
q(f"SELECT schema, COUNT(*) AS n FROM {SRC} GROUP BY schema ORDER BY n DESC")

             0       1     2        3           4          5          6   \
column_name  id  schema  name  aliases  birth_date  countries  addresses   

                      7          8       9       10           11       12  \
column_name  identifiers  sanctions  phones  emails  program_ids  dataset   

                     13         14           15  
column_name  first_seen  last_seen  last_change  


,schema,n
0,Person,1033337
1,Company,99100
2,LegalEntity,58988
3,Security,20530
4,CryptoWallet,13797
5,Organization,9941
6,Vessel,9120
7,Airplane,342
8,Asset,36
9,Address,19


In [3]:
q(f"""SELECT dataset, COUNT(*) AS n FROM {SRC}
      WHERE dataset ILIKE '%disqualif%' GROUP BY dataset ORDER BY n DESC LIMIT 5""")

,dataset,n
0,UK Companies House Disqualified Directors,6048
1,Brazil List of Individuals Disqualified from P...,545
2,Brazil Register of Persons Disqualified from S...,316
3,UK Companies House Disqualified Directors;UK F...,285
4,US FDA Clinical Investigator Disqualification ...,137


In [4]:
targets = q(f"""
SELECT id, schema, name, aliases, countries, sanctions, dataset,
       CASE WHEN schema IN ('Company', 'Organization', 'LegalEntity') THEN 'organisation'
            ELSE 'disqualified_person' END AS target_type
FROM {SRC}
WHERE schema IN ('Company', 'Organization', 'LegalEntity')
   OR dataset ILIKE '%disqualif%'
""")
targets["name_norm"] = targets["name"].map(normalise_name)
targets = targets[targets["name_norm"] != ""]

print(targets["target_type"].value_counts())
print("changed by normalising:", f"{(targets['name_norm'] != targets['name'].str.upper()).mean():.0%}")
print(targets[["name", "name_norm", "countries"]].sample(10, random_state=1).to_string(index=False))

OUT = ROOT / "data" / "processed" / "opensanctions_targets.parquet"
targets.to_parquet(OUT, index=False)
con.register("t", targets)
con.execute("CREATE OR REPLACE TABLE os_targets AS SELECT * FROM t")
con.unregister("t")
print("saved:", OUT.relative_to(ROOT), len(targets), "rows")

target_type
organisation           130506
disqualified_person      9971
Name: count, dtype: int64
changed by normalising: 60%
                                                                                                                                   name                        name_norm countries
ФЕДЕРАЛЬНОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ "ИСПРАВИТЕЛЬНАЯ КОЛОНИЯ № 6 УПРАВЛЕНИЯ ФЕДЕРАЛЬНОЙ СЛУЖБЫ ИСПОЛНЕНИЯ НАКАЗАНИЙ ПО АСТРАХАНСКОЙ ОБЛАСТИ"                             NO 6        ru
                                                                                                                           E-Z Pharmacy                      EZ PHARMACY        us
                                                                                                                        JACQUES A. FRYM                   JACQUES A FRYM        us
                                                                                                                     Timothy J. Scherwa                TIMOTHY